In [18]:
# =============================================================================
# CELL 1 — IMPORTS & CONFIGURATION (FISIK SPORT)
# =============================================================================

import pandas as pd
import numpy as np
import re
import math
import warnings
warnings.filterwarnings('ignore')

FILE_PATH = r'C:\Users\dadia\capstone-project\db\raw\FISIK SPORT.xlsx'
BRAND_ID = 'f0cd0235-ffd8-4ac3-b033-ae9c86d4f0e8'  # FISIK SPORT UUID

SUPABASE_URL = 'https://zzfghscdyvwecxrjzqrn.supabase.co'
SUPABASE_KEY = 'sb_secret_cVHnlsiAHiZXaV1vjbl0bA_W_mtBGh_'

PLATFORM_MAP = {
    'tiktok' : '4d324e89-b6c0-44c2-8aac-f821f6087b9e',
    'shopee' : '045dbc37-9740-44cd-99b7-03489159173a',
}

HOST_MAP = {
    'jeje':     11,    # JEJE (used in PIERO/MINERAL)
    'billy':    29,    # billy (used in PIERO/MINERAL)
    'wulan':    36,    # wulan (used in PIERO/MINERAL)
    'deva':     37,    # deva (used in PIERO/MINERAL)
    'gerson':   8,     # GERSON (used in PIERO/MINERAL)
    'mahen':    57,    # mahen (used in PIERO/MINERAL)
    'sopi':     55,    # sopi (used in PIERO/MINERAL)
    'hanif':    54,    # hanif (used in PIERO/MINERAL)
    'muti':     56,    # muti (used in PIERO/MINERAL)
    'michael':  51,    # michael (used in PIERO/MINERAL)
    'aliza':    50,    # aliza (used in PIERO/MINERAL)
    'dinda':    110,   # DINDA
    'tata':     64,    # tata (used in PIERO/MINERAL)
    'nadya':    105,   # NAYA
    'nurul':    129,   # NEW
    'mewa':     130,   # NEW
    'dini':     131,   # NEW
    'aprillia': 132,   # NEW
    'uci':      133,   # NEW
}

JUNK_HOST_VALUES = {'host live', 'host', 'nama', 'name', 'total', 'off', 'libur', '', 'nan'}
JUNK_PLATFORM_VALUES = {'platform', 'host', 'sesi', '', 'nan'}

print(f'✅ CELL 1 — FISIK SPORT configuration loaded')
print(f'   FILE: {FILE_PATH}')
print(f'   BRAND_ID: {BRAND_ID}')
print(f'   Hosts mapped: {len(HOST_MAP)}')

✅ CELL 1 — FISIK SPORT configuration loaded
   FILE: C:\Users\dadia\capstone-project\db\raw\FISIK SPORT.xlsx
   BRAND_ID: f0cd0235-ffd8-4ac3-b033-ae9c86d4f0e8
   Hosts mapped: 19


In [4]:
# =============================================================================
# CELL 2 — HELPER FUNCTIONS
# =============================================================================
# Run this after CELL 1. Defines all cleaning functions.
# =============================================================================

def find_header_row(df):
    """Find the row containing column headers."""
    for i in range(min(20, len(df))):
        row_values = [str(v).lower().strip() for v in list(df.iloc[i])]
        keywords = ['tanggal', 'jam', 'platform', 'host', 'revenue']
        matches = sum(1 for kw in keywords if any(kw in v for v in row_values))
        if matches >= 2:
            return i
    return None


def clean_date(val):
    """Convert various date formats to YYYY-MM-DD."""
    if pd.isna(val) or str(val).strip() in ('', 'nan', 'NaT'):
        return None
    if hasattr(val, 'strftime'):
        return val.strftime('%Y-%m-%d')
    
    s = str(val).strip()
    
    # Try ISO/pandas format first
    try:
        return pd.to_datetime(s).strftime('%Y-%m-%d')
    except:
        pass
    
    # Try Indonesian date format ("1 Januari 2026")
    MONTH_MAP = {
        'januari':'01', 'februari':'02', 'maret':'03', 'april':'04',
        'mei':'05', 'juni':'06', 'juli':'07', 'agustus':'08',
        'september':'09', 'oktober':'10', 'november':'11', 'desember':'12',
        # English month names as fallback
        'january':'01', 'february':'02', 'march':'03', 'may':'05',
        'june':'06', 'july':'07', 'august':'08', 'october':'10',
        'december':'12'
    }
    s_lower = s.lower()
    for month_name, month_num in MONTH_MAP.items():
        if month_name in s_lower:
            parts = re.findall(r'\d+', s_lower)
            if len(parts) >= 2:
                day = parts[0].zfill(2)
                year = parts[-1]
                if len(year) == 4 and int(year) < 2000:
                    year = '20' + year[2:]
                elif len(year) == 2:
                    year = '20' + year
                try:
                    return pd.to_datetime(f'{year}-{month_num}-{day}').strftime('%Y-%m-%d')
                except:
                    pass
    return None


def clean_time(val):
    """Normalize time ranges to HH:MM-HH:MM."""
    if pd.isna(val) or str(val).strip() in ('', 'nan'):
        return None
    s = str(val).strip().replace('.', ':').replace(' ', '')
    if '-' in s:
        parts = s.split('-')
        return f"{parts[0].strip()}-{parts[1].strip()}"
    return s


def clean_revenue(val):
    """Convert revenue strings to integers."""
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if s in ('', 'nan', '-', '0', 'Rp. 0'):
        return 0
    s = re.sub(r'[Rr][Pp][.\s]*', '', s)
    s = re.sub(r'[.\s]', '', s)
    s = s.replace(',', '.')
    try:
        return int(float(s))
    except:
        return 0


def clean_viewers(val):
    """Convert viewer/likes counts to integers."""
    if pd.isna(val):
        return 0
    s = str(val).strip().lower()
    if s in ('', 'nan', '-'):
        return 0
    if 'rb' in s:
        s = s.replace('rb', '').replace(',', '.').strip()
        try:
            return int(float(s) * 1000)
        except:
            pass
    if 'k' in s:
        s = s.replace('k', '').replace(',', '.').strip()
        try:
            return int(float(s) * 1000)
        except:
            pass
    s = re.sub(r'[,\s]', '', s)
    try:
        return int(float(s))
    except:
        return 0


def clean_host(val):
    """Clean host names."""
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if s in JUNK_HOST_VALUES:
        return None
    if re.match(r'^\d+[\.,]?\d*$', s):
        return None
    s = re.sub(r'\s*\(.*?\)', '', s)
    return s.strip()


def clean_platform(val):
    """Normalize platform names."""
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if s in JUNK_PLATFORM_VALUES:
        return None
    if 'tiktok' in s or s == 'tt':
        return 'tiktok'
    if 'shopee' in s:
        return 'shopee'
    return None


print('✅ CELL 2 — Helper functions defined')

✅ CELL 2 — Helper functions defined


In [12]:
# =============================================================================
# CELL 3 — READ & CLEAN ALL SHEETS
# =============================================================================
# Extracts and cleans data from every sheet in the Excel file.
# =============================================================================

def extract_sheet_data(df_raw, header_row, periode_num):
    """Extract and clean data from one sheet."""
    raw_cols = [str(c).lower().strip() for c in df_raw.iloc[header_row]]
    data = df_raw.iloc[header_row + 1:].copy()
    
    # Handle column count mismatch
    n_header = len(raw_cols)
    n_data = data.shape[1]
    if n_data > n_header:
        data = data.iloc[:, :n_header]
    
    data.columns = raw_cols[:data.shape[1]]
    
    # Map column names
    col_map = {}
    for c in data.columns:
        s = str(c).lower().strip()
        if 'tanggal' in s: col_map[c] = 'date'
        elif s in ('jam live', 'jam', 'time'): col_map[c] = 'time'
        elif 'platform' in s: col_map[c] = 'platform'
        elif 'host' in s: col_map[c] = 'host'
        elif 'rev' in s and 'tik' in s: col_map[c] = 'revenue_tiktok'
        elif 'rev' in s and 'shop' in s: col_map[c] = 'revenue_shopee'
        elif 'rev' in s: col_map[c] = 'revenue_shopee'
        elif 'view' in s and 'tik' in s: col_map[c] = 'viewers_tiktok'
        elif 'view' in s and 'shop' in s: col_map[c] = 'viewers_shopee'
        elif 'view' in s: col_map[c] = 'viewers_shopee'
        elif 'like' in s and 'tik' in s: col_map[c] = 'likes_tiktok'
        elif 'like' in s and 'shop' in s: col_map[c] = 'likes_shopee'
        elif 'like' in s: col_map[c] = 'likes_shopee'
        elif 'comment' in s: col_map[c] = 'comment'
    
    data = data.rename(columns=col_map)
    
    # Ensure required columns exist
    required = ['date','time','platform','host',
                'revenue_tiktok','viewers_tiktok','likes_tiktok',
                'revenue_shopee','viewers_shopee','likes_shopee']
    for col in required:
        if col not in data.columns:
            data[col] = np.nan
    
    # Clean all columns
    data['date'] = data['date'].apply(clean_date)
    data['time'] = data['time'].apply(clean_time)
    data['platform'] = data['platform'].apply(clean_platform)
    data['host'] = data['host'].apply(clean_host)
    
    for col in ['revenue_tiktok','revenue_shopee']:
        data[col] = data[col].apply(clean_revenue)
    for col in ['viewers_tiktok','viewers_shopee','likes_tiktok','likes_shopee']:
        data[col] = data[col].apply(clean_viewers)
    
    # Forward fill dates (merged cells)
    data['date'] = data['date'].ffill()
    data['platform'] = data['platform'].ffill()
    
    # Remove junk rows
    data = data[data['host'].notna()]
    data = data[data['date'].notna()]
    data = data[data['time'].notna()]
    
    data['periode_id'] = periode_num
    data = data.reset_index(drop=True)
    
    final_cols = ['date','time','platform','host',
                  'revenue_tiktok','viewers_tiktok','likes_tiktok',
                  'revenue_shopee','viewers_shopee','likes_shopee','periode_id']
    
    return data[[c for c in final_cols if c in data.columns]]


# ========== MAIN PROCESSING ==========
print('🔄 Reading FISIK SPORT.xlsx...')
xl = pd.ExcelFile(FILE_PATH)
print(f'   Found {len(xl.sheet_names)} sheets')

all_frames = []
skipped = []

for sheet_name in xl.sheet_names:
    # Extract period number
    match = re.search(r'(\d+)', sheet_name.upper().replace('PRIODE', 'PERIODE'))
    if not match:
        skipped.append(sheet_name)
        continue
    
    p_num = int(match.group(1))
    df_raw = pd.read_excel(FILE_PATH, sheet_name=sheet_name, header=None)
    header_row = find_header_row(df_raw)
    
    if header_row is None:
        print(f'   ⚠️  Period {p_num} ({sheet_name}): No header found, skipping')
        skipped.append(sheet_name)
        continue
    
    try:
        df_sheet = extract_sheet_data(df_raw, header_row, p_num)
        if len(df_sheet) > 0:
            all_frames.append(df_sheet)
            print(f'   ✅ Period {p_num:2d}: {len(df_sheet):3d} rows')
        else:
            print(f'   ⚠️  Period {p_num}: 0 rows after cleaning')
    except Exception as e:
        print(f'   ❌ Period {p_num}: Error - {str(e)[:80]}')
        skipped.append(sheet_name)

if all_frames:
    df = pd.concat(all_frames, ignore_index=True)
    df = df.sort_values(['periode_id', 'date']).reset_index(drop=True)
    print(f'\n✅ CELL 3 — Extraction complete')
    print(f'   Total rows: {len(df)}')
    print(f'   Periods: {sorted(df["periode_id"].unique())}')
    if skipped:
        print(f'   Skipped sheets: {skipped}')
else:
    print('\n❌ No data extracted! Check FILE_PATH and sheet names.')
    df = pd.DataFrame()

🔄 Reading FISIK SPORT.xlsx...
   Found 15 sheets
   ✅ Period  1:  64 rows
   ✅ Period  2:  61 rows
   ✅ Period  3:  60 rows
   ✅ Period  4:  61 rows
   ✅ Period  5:  61 rows
   ✅ Period  7:  61 rows
   ✅ Period  6:  60 rows
   ✅ Period  8:  62 rows
   ✅ Period  9:  60 rows
   ✅ Period 10:  60 rows
   ✅ Period 11:  60 rows
   ✅ Period 12:  60 rows
   ✅ Period 13:  60 rows
   ✅ Period 14:  50 rows
   ⚠️  Period 8 (PERIODE 8): No header found, skipping

✅ CELL 3 — Extraction complete
   Total rows: 840
   Periods: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14)]
   Skipped sheets: ['PERIODE 8']


In [13]:
# =============================================================================
# CELL 4 — DATA QUALITY CHECK
# =============================================================================
# Review BEFORE inserting into database!
# =============================================================================

if df.empty:
    print('❌ DataFrame is empty — fix CELL 3 before continuing!')
else:
    print('=' * 60)
    print('DATA QUALITY REPORT — FISIK SPORT')
    print('=' * 60)
    
    print(f'\nTotal rows: {len(df)}')
    
    print(f'\nMissing values:')
    print(df.isnull().sum())
    
    print(f'\nPlatforms:')
    print(df['platform'].value_counts(dropna=False))
    
    print(f'\nHosts (top 20):')
    print(df['host'].value_counts().head(20))
    
    print(f'\nDate range:')
    print(f'   From: {df["date"].min()}')
    print(f'   To:   {df["date"].max()}')
    
    print(f'\nRows per period:')
    print(df['periode_id'].value_counts().sort_index())
    
    # Check unmapped hosts
    unmapped = df[~df['host'].isin(HOST_MAP.keys())]['host'].value_counts()
    if len(unmapped) > 0:
        print(f'\nUNMAPPED HOSTS (will get NULL host_id):')
        for host, count in unmapped.items():
            print(f'   {host}: {count} sessions')
        print('\n Add these to HOST_MAP in CELL 1!')
    else:
        print('\n✅ All hosts are mapped!')
    
    print(f'\n Revenue Shopee:')
    print(f'   Total: {df["revenue_shopee"].sum():,.0f}')
    print(f'   Mean:  {df["revenue_shopee"].mean():,.0f}')
    print(f'   Zeros: {(df["revenue_shopee"] == 0).sum()} rows')

DATA QUALITY REPORT — FISIK SPORT

Total rows: 840

Missing values:
date                0
time                0
platform          170
host                0
revenue_tiktok      0
viewers_tiktok      0
likes_tiktok        0
revenue_shopee      0
viewers_shopee      0
likes_shopee        0
periode_id          0
dtype: int64

Platforms:
platform
shopee    670
None      170
Name: count, dtype: int64

Hosts (top 20):
host
wulan      160
mahen      158
gerson     146
sopi       116
muti       110
michael    110
deva        14
jeje         5
uci          4
tata         4
billy        3
nurul        2
mewa         2
nadya        1
hanif        1
dini         1
aprilia      1
aliza        1
dinda        1
Name: count, dtype: int64

Date range:
   From: 2025-03-03
   To:   2026-04-30

Rows per period:
periode_id
1     64
2     61
3     60
4     61
5     61
6     60
7     61
8     62
9     60
10    60
11    60
12    60
13    60
14    50
Name: count, dtype: int64

UNMAPPED HOSTS (will get NULL host

In [14]:
# =============================================================================
# FIX: Forward fill platform for merged cells + add aprilia
# =============================================================================

# Convert None to NaN first, then ffill
df['platform'] = df['platform'].replace({None: np.nan})
df['platform'] = df['platform'].ffill()

# Also fix the aprilia/aprillia typo
HOST_MAP['aprilia'] = 132

print(f'Platforms after fix:')
print(df['platform'].value_counts(dropna=False))
print(f'\nUnmapped hosts:')
unmapped = df[~df['host'].isin(HOST_MAP.keys())]['host'].value_counts()
print(unmapped if len(unmapped) > 0 else '✅ All mapped!')

Platforms after fix:
platform
shopee    840
Name: count, dtype: int64

Unmapped hosts:
✅ All mapped!


In [15]:
# =============================================================================
# CELL 5 — MAP TO SUPABASE SCHEMA
# =============================================================================
# Renames columns and converts types for Supabase insert.
# =============================================================================

if df.empty:
    print('❌ DataFrame is empty!')
else:
    df_insert = pd.DataFrame()
    
    # Map platform and host
    df_insert['platform_id'] = df['platform'].map(PLATFORM_MAP)
    df_insert['host_id'] = df['host'].map(HOST_MAP)
    
    # Direct mappings
    df_insert['date'] = df['date']
    df_insert['time'] = df['time']
    df_insert['brand_id'] = BRAND_ID
    df_insert['revenue_shopee'] = df['revenue_shopee']
    df_insert['viewers_shopee'] = df['viewers_shopee']
    df_insert['likes_shopee'] = df['likes_shopee']
    df_insert['revenue_tiktok'] = df['revenue_tiktok']
    df_insert['viewers_tiktok'] = df['viewers_tiktok']
    df_insert['likes_tiktok'] = df['likes_tiktok']
    df_insert['period_id'] = df['periode_id']
    
    # Convert to integers
    int_cols = ['revenue_shopee','viewers_shopee','likes_shopee',
                'revenue_tiktok','viewers_tiktok','likes_tiktok']
    for col in int_cols:
        df_insert[col] = df_insert[col].fillna(0).astype(int)
    
    df_insert['host_id'] = df_insert['host_id'].astype('Int64')
    
    print(f'✅ CELL 5 — Data mapped to schema')
    print(f'   Columns: {list(df_insert.columns)}')
    print(f'   Rows: {len(df_insert)}')
    print(f'\n   Sample:')
    print(df_insert.head(3).to_string())

✅ CELL 5 — Data mapped to schema
   Columns: ['platform_id', 'host_id', 'date', 'time', 'brand_id', 'revenue_shopee', 'viewers_shopee', 'likes_shopee', 'revenue_tiktok', 'viewers_tiktok', 'likes_tiktok', 'period_id']
   Rows: 840

   Sample:
                            platform_id  host_id        date         time                              brand_id  revenue_shopee  viewers_shopee  likes_shopee  revenue_tiktok  viewers_tiktok  likes_tiktok  period_id
0  045dbc37-9740-44cd-99b7-03489159173a       11  2025-03-03  10:00-12:00  f0cd0235-ffd8-4ac3-b033-ae9c86d4f0e8         3443460             739             0               0               0             0          1
1  045dbc37-9740-44cd-99b7-03489159173a       29  2025-03-03  22:00-00:00  f0cd0235-ffd8-4ac3-b033-ae9c86d4f0e8         3469618            1051             0               0               0             0          1
2  045dbc37-9740-44cd-99b7-03489159173a       11  2025-03-04  10:00-12:00  f0cd0235-ffd8-4ac3-b033-ae9c86d4f0e8  

In [16]:
# =============================================================================
# CELL 6 — CONNECT TO SUPABASE
# =============================================================================

from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ CELL 6 — Connected to Supabase')

✅ CELL 6 — Connected to Supabase


In [19]:
# =============================================================================
# CELL 7 — CHECK EXISTING DATA
# =============================================================================

check = supabase.table('live_sessions') \
    .select('id', count='exact') \
    .eq('brand_id', BRAND_ID) \
    .execute()

print(f'🔍 Found {check.count} existing rows for FISIK SPORT in live_sessions')

if check.count > 0:
    print('   ⚠️  These will be deleted in CELL 7B')
else:
    print('   ✅ No existing data — safe to proceed to CELL 8')

🔍 Found 0 existing rows for FISIK SPORT in live_sessions
   ✅ No existing data — safe to proceed to CELL 8


In [20]:
# =============================================================================
# CELL 8 — INSERT DATA
# =============================================================================

def replace_nan(val):
    if val is None:
        return None
    try:
        if pd.isna(val):
            return None
    except:
        pass
    if isinstance(val, (np.integer,)):
        return int(val)
    return val

records = []
for _, row in df_insert.iterrows():
    record = {col: replace_nan(row[col]) for col in df_insert.columns}
    records.append(record)

print(f'📤 Inserting {len(records)} rows...')

BATCH_SIZE = 200
success = 0
failed = 0

for i in range(0, len(records), BATCH_SIZE):
    batch = records[i:i + BATCH_SIZE]
    try:
        supabase.table('live_sessions').insert(batch).execute()
        success += len(batch)
        print(f'   ✅ Batch {i//BATCH_SIZE+1}: {len(batch)} rows (total: {success})')
    except Exception as e:
        failed += len(batch)
        print(f'   ❌ Batch {i//BATCH_SIZE+1} failed: {str(e)[:100]}')

print(f'\n✅ Success: {success} | ❌ Failed: {failed}')

📤 Inserting 840 rows...
   ✅ Batch 1: 200 rows (total: 200)
   ✅ Batch 2: 200 rows (total: 400)
   ✅ Batch 3: 200 rows (total: 600)
   ✅ Batch 4: 200 rows (total: 800)
   ✅ Batch 5: 40 rows (total: 840)

✅ Success: 840 | ❌ Failed: 0


In [21]:
# =============================================================================
# CELL 9 — VERIFY
# =============================================================================

verify = supabase.table('live_sessions') \
    .select('period_id, revenue_shopee') \
    .eq('brand_id', BRAND_ID) \
    .execute()

if verify.data:
    vdf = pd.DataFrame(verify.data)
    summary = vdf.groupby('period_id').agg(
        sessions=('period_id','count'),
        total_rev=('revenue_shopee','sum')
    ).sort_index()
    print(f'✅ {len(vdf)} rows verified')
    print(summary.to_string())
else:
    print('❌ No data found!')

✅ 840 rows verified
           sessions  total_rev
period_id                     
1                64   67781795
2                61   30703213
3                60  223129179
4                61  210643906
5                61   36428646
6                60  309100235
7                61  116910300
8                62  182516550
9                60  130101425
10               60  118185435
11               60   82445600
12               60   24029715
13               60   91269574
14               50   36005167
